# Week 2 · C: learn to drive in the dream

The controller is a linear layer, 867 parameters. CMA-ES evolves it using **only rewards M imagines**. Every tenth generation the current best guess also drives the real track, purely so we can watch the gap between what the dream promises and what reality delivers. Those real scores never reach the optimiser.

**A note on fidelity.** In the 2018 paper, the CarRacing controller was trained in the real environment; training wholly in the dream was the VizDoom experiment. This notebook does the dream version on CarRacing because that is what the plan asks for and it is the more instructive test. It is also harder. Expect a gap, and treat measuring it as the point.

About 30–45 minutes on a T4. Dream scoring is batched on the GPU; the occasional real drive is the slow part.

In [ ]:
# --- Setup: runs on Colab or locally (from the notebooks/ folder) -----------
REPO_URL = "https://github.com/vamsi1905/world-models-lab.git"   # <- edit once
USE_DRIVE = True        # keep data and checkpoints on Google Drive across disconnects

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("WM_SMOKE") == "1"      # tiny sizes, used only to test the notebook
if IN_COLAB:
    if not os.path.exists("/content/world-models-lab"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/world-models-lab"], check=True)
    os.chdir("/content/world-models-lab")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "swig"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK = "/content/drive/MyDrive/world-models-lab"
    else:
        WORK = "/content/work"
else:
    sys.path.insert(0, os.path.abspath(".."))
    WORK = os.path.abspath(os.environ.get("WM_WORK", "../work"))
os.makedirs(WORK, exist_ok=True)

import numpy as np, torch, matplotlib.pyplot as plt
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| work dir:", WORK, "| smoke test:", SMOKE)

In [ ]:
from wmlab.carracing.vae import ConvVAE
from wmlab.carracing.mdnrnn import MDNRNN
from wmlab.carracing.controller import train_in_dream, real_rollout, dream_video_frames, side_by_side_video, N_PARAMS
vae = ConvVAE().to(DEVICE); vae.load_state_dict(torch.load(f"{WORK}/vae.pt", map_location=DEVICE)); vae.eval()
mdn = MDNRNN().to(DEVICE); mdn.load_state_dict(torch.load(f"{WORK}/mdnrnn.pt", map_location=DEVICE)); mdn.eval()
lat = np.load(f"{WORK}/latents.npz")
z0_pool = torch.from_numpy(np.stack([lat[k][0] for k in lat.files if k.startswith("mu_")])).to(DEVICE)
print("controller parameters:", N_PARAMS, "| dream start frames:", len(z0_pool))

## 1 · Evolve in the dream
`temperature` is the lever Ha & Schmidhuber used against cheating: above 1.0 the dream gets noisier, so exploits that depend on M's exact quirks stop paying.

In [ ]:
import json
TEMP = 1.15
es, log = train_in_dream(vae, mdn, z0_pool,
                         generations=3 if SMOKE else 150, popsize=8 if SMOKE else 64,
                         dream_steps=40 if SMOKE else 300, temperature=TEMP, repeats=2 if SMOKE else 8,
                         real_every=2 if SMOKE else 10, real_steps=40 if SMOKE else 300, device=DEVICE)
np.save(f"{WORK}/controller_t{TEMP}.npy", es.mean); json.dump(log, open(f"{WORK}/cma_log_t{TEMP}.json", "w"))
g = [r["gen"] for r in log]; rg = [r["gen"] for r in log if "real_of_mean" in r]
plt.figure(figsize=(7, 3.5))
plt.plot(g, [r["dream_mean"] for r in log], label="dream: population mean")
plt.plot(g, [r["dream_best"] for r in log], alpha=.5, label="dream: best")
plt.plot(rg, [r["real_of_mean"] for r in log if "real_of_mean" in r], "ko-", label="real track: CMA mean")
plt.xlabel("generation"); plt.ylabel(f"reward over {300 if not SMOKE else 40} steps"); plt.legend(); plt.grid(alpha=.3)
plt.title("The dream-to-reality gap"); plt.savefig(f"{WORK}/dream_vs_real.png", dpi=150, bbox_inches="tight"); plt.show()

**Reading the chart.** Dream score rising while the real score stays flat means the controller has found a hole in M rather than a way to drive. Try `TEMP = 1.5` and rerun. If both rise together, you have the result the plan promised.

## 2 · Score it properly
The standard CarRacing measure: full 1,000-step episodes on unseen tracks. The paper's full-strength agent scored about 906; a random policy scores well below zero. Anything clearly positive from a dream-only controller is a result.

In [ ]:
seeds = range(2 if SMOKE else 10)
scores = [real_rollout(vae, mdn, es.mean, seed=50_000 + s, max_steps=60 if SMOKE else 1000, device=DEVICE)[0] for s in seeds]
print("real-track scores:", np.round(scores, 1), "| mean", np.mean(scores).round(1))

## 3 · The video
Left: the controller driving in M's imagination, decoded through V. Right: the same controller on a real track. This is the cold open.

In [ ]:
n = 60 if SMOKE else 400
from wmlab.carracing.vae import to_tensor
_, real_frames = real_rollout(vae, mdn, es.mean, seed=0, max_steps=n, record=True, device=DEVICE)
with torch.no_grad():
    z_start = vae.encode(to_tensor(real_frames[:1], DEVICE))[0][0]     # dream from the same first frame
dream = dream_video_frames(vae, mdn, es.mean, z_start, steps=n, temperature=1.0, device=DEVICE)
out = side_by_side_video(dream, real_frames, f"{WORK}/dream_vs_real.mp4")
print("saved", out)
if IN_COLAB:
    from IPython.display import Video; display(Video(out, embed=True, width=520))

## 4 · When it does not drive

Work down this list in order; each is cheaper than the next.

1. **Hotter dream.** Retrain C at `TEMP = 1.5`. If the real score improves, M was being exploited.
2. **Longer M training** or `seq_len=200`. A dream that drifts after 50 steps cannot teach a 300-step skill.
3. **Iterate, as the paper did for harder tasks.** Collect 100 new rollouts with the trained controller plus noise, retrain M on old plus new data, retrain C. Random driving rarely completes a corner, so M has never seen what competent driving looks like.
4. **The paper's own route.** Keep V and M, but score C in the real environment. Slower (hours, not minutes), and it tells you whether the problem is the dream or the features.

Log whichever you try, with the chart. A documented gap is as publishable as a clean success.